# 04 — Train YOLO26m

Phụ trách: **Thành viên 1**. Pipeline Ultralytics YOLO26m trên dataset biển báo Việt Nam, không crop từng biển. Checkpoint chọn theo mAP50-95 trên val. Mosaic và mixup chỉ chạy khi `ONLINE = True`.

## 0) Bootstrap (local / Kaggle / Colab)

- **Local**: notebook nằm trong repo nên dùng luôn repo hiện tại (không đụng vào git của bạn).
- **Kaggle / Colab**: clone repo về `/kaggle/working`, lấy đúng nhánh rồi cài thư viện. Trên Kaggle bật **Internet: On**.
- Repo private: xem `README.md` mục 12.8 (token lưu trong Kaggle Secrets).

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/ThanhBinh2006/VN-TSD-Voice-Alert.git"
REPO_DIR = "VN-TSD-Voice-Alert"
REPO_BRANCH = "main"   # Nhánh hiện tại (hoặc "test-train-model" sau khi merge)

ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB = "google.colab" in sys.modules

def sh(cmd: str, check: bool = True) -> int:
    print(">>", cmd)
    return subprocess.run(cmd, shell=True, check=check).returncode

# 1. Tìm repo đang chứa notebook này (local)
local_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "ts_alert").is_dir() and (p / "configs" / "shared.yaml").exists()), None)

if local_root is not None:
    repo_path = local_root
    print("Dùng repo hiện tại:", repo_path)
else:
    # 2. Chưa có repo: clone rồi lấy đúng nhánh (chỉ làm trong môi trường cloud)
    assert REPO_URL != "TODO_URL", "Điền REPO_URL ở ô này trước khi chạy trên Kaggle/Colab."
    base = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
    repo_path = base / REPO_DIR
    if not (repo_path / ".git").exists():
        sh(f'git clone --branch {REPO_BRANCH} {REPO_URL} "{repo_path}"')
    git = f'git -C "{repo_path}"'
    sh(f"{git} fetch origin {REPO_BRANCH}")
    sh(f"{git} checkout -B {REPO_BRANCH} origin/{REPO_BRANCH}")
    sh(f"{git} log --oneline -1")

os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
print("cwd:", Path.cwd())

# Cài thư viện: chỉ trên Kaggle/Colab. Local thì dùng môi trường đã cài bằng requirements.txt.
if ON_KAGGLE or ON_COLAB:
    sh('pip install -q -e ".[detect]"')
    sh("pip install -q wandb==0.26.1")

import torch
import ts_alert
print(f"Bootstrap OK. PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")

## 1) Đường dẫn data/processed

Cache offline dùng chung: ảnh letterbox và nhãn YOLO, không crop từng biển. Trên Kaggle gắn dataset đã lưu từ notebook 02 bằng Add Input rồi điền vào `CANDIDATES`.

In [ ]:
# Tự động quét và phát hiện dataset đã gắn trong /kaggle/input/ hoặc local
import glob
from pathlib import Path

found_roots = []
for p in glob.glob('/kaggle/input/**/images/train', recursive=True):
    found_roots.append(str(Path(p).parent))
for p in glob.glob('/kaggle/input/**/data.yaml', recursive=True):
    found_roots.append(str(Path(p).parent))

CANDIDATES = [
    *found_roots,
    *glob.glob('/kaggle/input/**/data/processed', recursive=True),
    *glob.glob('/kaggle/input/**/processed', recursive=True),
    *glob.glob('/kaggle/input/*', recursive=False),
    'data/processed',
    '/kaggle/working/data/processed',
]

def _ready(path: Path) -> bool:
    return (path / 'images' / 'train').exists() or (path / 'data.yaml').exists()

_found = next((c for c in CANDIDATES if c and _ready(Path(c))), None)
if _found is None:
    raise FileNotFoundError(
        '❌ Không tìm thấy thư mục dataset (cần có images/train hoặc data.yaml).\n'
        '👉 Trên Kaggle: Hãy bấm nút "+ Add Input" (ở góc trên bên phải màn hình), tìm và gắn dataset đã tiền xử lý.\n'
        '👉 Nếu chạy local: Hãy chạy notebook 02 (tiền xử lý offline) trước.'
    )

PROCESSED_ROOT = str(Path(_found).resolve())
print('[OK] Processed root:', PROCESSED_ROOT)

# Tạo symlink data/processed -> PROCESSED_ROOT nếu đường dẫn khác nhau
target = Path('data/processed')
if PROCESSED_ROOT != str(target.resolve()) and Path(PROCESSED_ROOT).exists():
    if not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        try:
            target.symlink_to(Path(PROCESSED_ROOT), target_is_directory=True)
            print('[OK] Đã tạo symlink data/processed ->', PROCESSED_ROOT)
        except Exception:
            pass


## 2) Cấu hình

Sửa biến ở đây để thử nghiệm các siêu tham số:
- **Optimization**: optimizer (`auto`, `SGD`, `Adam`, `AdamW`), lr, momentum, weight_decay, warmup_epochs, warmup_momentum, warmup_bias_lr.
- **Data Augmentation**: mosaic, mixup, scale, translate, hsv_h, hsv_s, hsv_v, occlusion_p, occlusion_ratio (chỉ kích hoạt khi `ONLINE = True`).
- **Batch là TỔNG của cả hai GPU**, không phải mỗi GPU. Biển báo giao thông mang tính định hướng nên mặc định `hflip = False`.

Các giá trị dùng chung cho cả hai pipeline (epochs, batch train, batch eval, early stopping, metric chính, seed) được **đọc từ `configs/shared.yaml`** để hai notebook không lệch nhau. Muốn đổi thì sửa `shared.yaml` và báo cả nhóm.

In [ ]:
from ts_alert.config.resolve import resolve_config
import uuid

PIPELINE = "yolo"
_shared = resolve_config(None, None)         # chỉ đọc configs/shared.yaml
_pipe = resolve_config(PIPELINE, None)       # shared + configs/pipelines/yolo.yaml

# --- 1. Huấn luyện cơ bản & Weights & Biases ---
LR = float(_pipe["train"].get("lr", 0.01))
WEIGHTS = _pipe["model"]["pretrained"]         # vd. yolo26m.pt
REQUIRE_WANDB = True                         # Bật W&B; tự fallback local CSV/JSON/PNG nếu thiếu WANDB_API_KEY

# --- 2. Optimization Hyperparameters (Nhóm B) ---
OPTIMIZER = _pipe["train"].get("optimizer", "auto")              # auto, SGD, Adam, AdamW
MOMENTUM = float(_pipe["train"].get("momentum", 0.937))          # Momentum của optimizer (SGD / Adam beta1)
WEIGHT_DECAY = float(_pipe["train"].get("weight_decay", 0.0005)) # Regularization L2 weight decay
WARMUP_EPOCHS = float(_pipe["train"].get("warmup_epochs", 3.0))  # Số epoch warmup
WARMUP_MOMENTUM = float(_pipe["train"].get("warmup_momentum", 0.8)) # Momentum trong warmup
WARMUP_BIAS_LR = float(_pipe["train"].get("warmup_bias_lr", 0.1))   # Learning rate warmup cho bias

# --- 3. Data Augmentation Hyperparameters (Nhóm A - chỉ kích hoạt khi ONLINE = True) ---
ONLINE = bool(_pipe["online_preprocess"].get("enabled", False))    # True = bật augmentation online khi train
HFLIP = bool(_pipe["online_preprocess"].get("hflip", False))
MOSAIC = float(_pipe["online_preprocess"].get("mosaic", 1.0))       # Xác suất sử dụng Mosaic (0.0 - 1.0)
MIXUP = float(_pipe["online_preprocess"].get("mixup", 0.15))        # Xác suất sử dụng MixUp (0.0 - 1.0)
SCALE = float(_pipe["online_preprocess"].get("scale", 0.5))         # Mức độ thay đổi tỷ lệ ảnh (+/- fraction)
TRANSLATE = float(_pipe["online_preprocess"].get("translate", 0.1)) # Mức độ dịch chuyển (+/- fraction)
HSV_H = float(_pipe["online_preprocess"].get("hsv_h", 0.015))       # Biến đổi sắc độ Hue (0.0 - 1.0)
HSV_S = float(_pipe["online_preprocess"].get("hsv_s", 0.7))         # Biến đổi độ bão hòa Saturation (0.0 - 1.0)
HSV_V = float(_pipe["online_preprocess"].get("hsv_v", 0.4))         # Biến đổi độ sáng Brightness (0.0 - 1.0)
OCCLUSION_P = float(_pipe["online_preprocess"].get("occlusion_p", 0.30))     # Xác suất che khuất box
OCCLUSION_RATIO = float(_pipe["online_preprocess"].get("occlusion_ratio", 0.30)) # Tỉ lệ độ bị che phủ

# --- 4. Dùng chung, đọc từ configs/shared.yaml. Hai pipeline PHẢI giống nhau ---
SEED = _shared["seed"]
EPOCHS = int(_shared["train"]["epochs"])
IMGSZ = _shared["data"]["image_size"]
BATCH_SIZE = _shared["train"]["batch_size"]                      # TỔNG trên mọi GPU
EARLY_STOPPING_PATIENCE = _shared["train"]["early_stopping_patience"]
EVAL_BATCH_SIZE = _shared["eval"]["batch_size"]
USE_AMP = _shared["train"]["amp"]
PRIMARY_METRIC = _shared["train"]["primary_metric"]

# --- 5. W&B Sweep (đọc từ configs/shared.yaml & configs/pipelines/yolo.yaml) ---
_sweep = _pipe.get("sweep") or {}
SWEEP_ENABLED = True                                              # True = chạy tools.run_sweep thay vì 1 run đơn
SWEEP_METHOD = str(_sweep.get("method", "bayes"))
SWEEP_COUNT = int(_sweep.get("count", 10))

# Tên run: {pipeline}_s{seed}_ep{epochs}_bs{batch}[_online]_{hash8}
suffix = "_online" if ONLINE else ""
RUN_NAME = f"{PIPELINE}_s{SEED}_ep{EPOCHS}_bs{BATCH_SIZE}{suffix}_{uuid.uuid4().hex[:8]}"
print("RUN_NAME =", RUN_NAME)


## 3) Resolve config

Ghép biến ở trên với `configs/shared.yaml` và file pipeline. Ghi file overrides để lệnh train bên dưới dùng đúng cấu hình này.

In [ ]:
from ts_alert.config.resolve import resolve_config, get_num_classes
from ts_alert.utils.io import write_yaml

OVERRIDES = {
    "seed": SEED,
    "model": {"pretrained": WEIGHTS},
    "train": {
        "epochs": EPOCHS,
        "lr": LR,
        "optimizer": OPTIMIZER,
        "momentum": MOMENTUM,
        "weight_decay": WEIGHT_DECAY,
        "warmup_epochs": WARMUP_EPOCHS,
        "warmup_momentum": WARMUP_MOMENTUM,
        "warmup_bias_lr": WARMUP_BIAS_LR,
        "batch_size": BATCH_SIZE,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "amp": USE_AMP,
        "require_wandb": REQUIRE_WANDB,
        "run_name": RUN_NAME,
        "primary_metric": PRIMARY_METRIC,
    },
    "eval": {"batch_size": EVAL_BATCH_SIZE, "auto_eval_test": True, "split": "test"},
    "online_preprocess": {
        "enabled": ONLINE,
        "hflip": HFLIP,
        "mosaic": MOSAIC,
        "mixup": MIXUP,
        "scale": SCALE,
        "translate": TRANSLATE,
        "hsv_h": HSV_H,
        "hsv_s": HSV_S,
        "hsv_v": HSV_V,
        "occlusion_p": OCCLUSION_P,
        "occlusion_ratio": OCCLUSION_RATIO,
    },
    "data": {"root": PROCESSED_ROOT, "image_size": IMGSZ},
}

RUNTIME = "configs/runtime/kaggle.yaml" if ON_KAGGLE else "configs/runtime/local.yaml"
cfg = resolve_config(PIPELINE, RUNTIME, OVERRIDES)

OVERRIDES_PATH = Path("outputs") / "_notebook" / f"overrides_{PIPELINE}.yaml"
write_yaml(OVERRIDES, OVERRIDES_PATH)

train = cfg["train"]
op = cfg["online_preprocess"]
print("=== Cấu hình đã resolve ===")
print("Run name        :", RUN_NAME)
print("Pipeline        :", cfg["pipeline"])
print("Số lớp          :", get_num_classes(cfg))
print("Epochs          :", train["epochs"])
print("Optimizer       :", f"{train.get('optimizer')} (lr={train.get('lr')}, momentum={train.get('momentum')}, wd={train.get('weight_decay')})")
print("Warmup          :", f"epochs={train.get('warmup_epochs')}, momentum={train.get('warmup_momentum')}, bias_lr={train.get('warmup_bias_lr')}")
print("Primary metric  :", train["primary_metric"])
print("Batch total     :", train["batch_size"], "(YOLO truyền nguyên tổng, Ultralytics tự chia GPU)")
print("Eval batch total:", cfg["eval"]["batch_size"])
print("AMP             :", train["amp"])
print("W&B required    :", train["require_wandb"])
print("Sweep mode      :", SWEEP_ENABLED, f"(method={SWEEP_METHOD}, count={SWEEP_COUNT})")
print("Online Augment  :", op["enabled"])
if op["enabled"]:
    print(f"  - Mosaic={op.get('mosaic')}, Mixup={op.get('mixup')}, Scale={op.get('scale')}, Translate={op.get('translate')}")
    print(f"  - HSV=[h:{op.get('hsv_h')}, s:{op.get('hsv_s')}, v:{op.get('hsv_v')}], Occlusion=[p:{op.get('occlusion_p')}, ratio:{op.get('occlusion_ratio')}], HFlip={op.get('hflip')}")
print("Data root       :", cfg["data"]["root"])
print("Overrides file  :", OVERRIDES_PATH)


## 4) Thiết lập & Kiểm tra kết nối W&B

Khi `REQUIRE_WANDB = True`, quá trình huấn luyện sẽ vừa chạy vừa log trực tiếp theo từng epoch (loss, mAP50-95, mAP50, mAP75, AR100, precision, recall, F1, `system/learning_rate`) kèm bộ biểu đồ và kết quả chuẩn sau train (`training_curves.png`, `pr_curve.png`, `eval_comparison.png`, `confusion_matrix_test.png`, `confusion_matrix_val.png`, `metrics_test.json`, `metrics_val.json`, `metrics.json`) lên Weights & Biases.
- **Trên Kaggle**: tự động lấy `WANDB_API_KEY` từ biến môi trường (do `kaggle_push` truyền lên từ `.env`) hoặc từ Kaggle Secrets (`Add-ons → Secrets → WANDB_API_KEY`).
- **Ở máy Local**: đọc trực tiếp `WANDB_API_KEY` từ file `.env`.

In [ ]:
import os
from ts_alert.pipelines.tracking import load_wandb_env

if REQUIRE_WANDB or SWEEP_ENABLED:
    load_wandb_env(cfg)
    proj = cfg.get("project") or {}
    api_key = os.environ.get("WANDB_API_KEY", "").strip()
    if api_key:
        try:
            import wandb
            wandb.login(key=api_key, relogin=True)
            print(f"[OK] Đã sẵn sàng đồng bộ W&B trực tiếp vào project: {proj.get('wandb_project', 'TSD-Voice-Alert')}")
        except Exception as exc:
            print(f"[!] Không thể đăng nhập W&B ({exc}); tự động fallback vẽ biểu đồ local.")
    else:
        if SWEEP_ENABLED:
            raise RuntimeError("SWEEP_ENABLED = True nhưng chưa có WANDB_API_KEY trong .env hoặc Kaggle Secrets.")
        print("[!] Chưa có WANDB_API_KEY; huấn luyện vẫn chạy và lưu đầy đủ biểu đồ local (CSV/JSON/PNG).")
else:
    print("REQUIRE_WANDB = False: chỉ lưu biểu đồ local.")


## 5) Train (hoặc chạy W&B Sweep)

- Nếu `SWEEP_ENABLED = False`: Ultralytics tự quản lý DDP đa GPU khi `num_gpus > 1` (`device=[0, 1]`), chạy trực tiếp bằng `python -m tools.run_train` (không bọc qua `torchrun`).
- Nếu `SWEEP_ENABLED = True`: chạy W&B Sweep (`python -m tools.run_sweep`) với không gian tìm kiếm từ `configs/pipelines/yolo.yaml` (không chạm các khóa bất biến trong `configs/shared.yaml`).

In [ ]:
import subprocess

if SWEEP_ENABLED:
    cmd = [
        "python", "-m", "tools.run_sweep",
        "--pipeline", PIPELINE,
        "--runtime", RUNTIME,
        "--overrides", str(OVERRIDES_PATH),
        "--data-root", PROCESSED_ROOT,
        "--count", str(SWEEP_COUNT),
        "--method", SWEEP_METHOD,
        "--online" if ONLINE else "--no-online",
    ]
else:
    cmd = [
        "python", "-m", "tools.run_train",
        "--pipeline", PIPELINE,
        "--runtime", RUNTIME,
        "--run-name", RUN_NAME,
        "--overrides", str(OVERRIDES_PATH),
        "--data-root", PROCESSED_ROOT,
        "--online" if ONLINE else "--no-online",
    ]
print(">>", " ".join(cmd))
subprocess.run(cmd, check=True)


## 6) Tìm thư mục kết quả của run vừa train

In [ ]:
from pathlib import Path
import json
from ts_alert.config.resolve import resolve_path

out_root = resolve_path(cfg["paths"]["outputs_dir"]) / PIPELINE
if SWEEP_ENABLED:
    runs = sorted(out_root.glob(f"{PIPELINE}_sweep_*"), key=lambda p: p.stat().st_mtime)
    scored = []
    for p in runs:
        sp = p / "summary.json"
        if sp.is_file():
            val = json.loads(sp.read_text(encoding="utf-8")).get("best_metric")
            if val is not None:
                scored.append((float(val), p))
    if scored:
        RUN_DIR = max(scored, key=lambda item: item[0])[1]
    else:
        assert runs, f"Không thấy run sweep nào trong {out_root}"
        RUN_DIR = runs[-1]
else:
    runs = sorted(out_root.glob(f"{RUN_NAME}*"))
    assert runs, f"Không thấy run nào khớp {RUN_NAME}"
    RUN_DIR = runs[-1]

print("RUN_DIR =", RUN_DIR)
summary_path = RUN_DIR / "summary.json"
if summary_path.is_file():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print(
        f"[BEST CHECKPOINT] Epoch {summary.get('best_epoch')} | "
        f"{summary.get('primary_metric')} = {summary.get('best_metric')}"
    )

## 7) Đường cong huấn luyện & Biểu đồ đánh giá

Pipeline tự động chuẩn hóa log vào `results.csv`, `metrics_history.json`, `summary.json`, vẽ biểu đồ huấn luyện `training_curves.png` (Loss, Val Detection Metrics kèm mốc BEST, P/R/F1, Learning Rate) và các biểu đồ đánh giá sau train từ best checkpoint (`pr_curve.png`, `eval_comparison.png`, `confusion_matrix_test.png`, `confusion_matrix_val.png`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from ts_alert.pipelines.tracking import plot_training_curves

results_csv = RUN_DIR / "results.csv"
curves_png = RUN_DIR / "training_curves.png"
if results_csv.exists():
    log = pd.read_csv(results_csv)
    log.columns = [c.strip() for c in log.columns]
    print(f"Đã đọc {len(log)} epoch từ {results_csv.name}")
    if "is_best" in log.columns and (log["is_best"] == 1).any():
        best_row = log.loc[log["is_best"] == 1].iloc[-1]
        print("Dòng BEST:", best_row.to_dict())
    if not curves_png.is_file():
        best_ep = int(best_row["epoch"]) if "is_best" in log.columns and (log["is_best"] == 1).any() else None
        plot_training_curves(log.to_dict(orient="records"), curves_png, best_epoch=best_ep, primary_metric=PRIMARY_METRIC)
else:
    print(f"Chưa tìm thấy results.csv tại: {RUN_DIR}")

cm_test = RUN_DIR / "confusion_matrix_test.png"
if not cm_test.is_file():
    cm_test = RUN_DIR / "confusion_matrix.png"

eval_charts = [
    ("Đường cong huấn luyện (Loss, mAP/AR, P/R/F1, Learning Rate)", curves_png),
    ("Đường cong PR & F1-Confidence (Tìm conf* tối ưu trên Test)", RUN_DIR / "pr_curve.png"),
    ("So sánh chỉ số trên tập Test vs Val (tại conf* tối ưu)", RUN_DIR / "eval_comparison.png"),
    ("Ma trận nhầm lẫn trên tập Test (Confusion Matrix - Test)", cm_test),
    ("Ma trận nhầm lẫn trên tập Val (Confusion Matrix - Val)", RUN_DIR / "confusion_matrix_val.png"),
]

for title, img_path in eval_charts:
    if img_path.is_file():
        fig, ax = plt.subplots(figsize=(12, 8))
        with Image.open(img_path) as img:
            ax.imshow(img.copy())
        ax.set_title(title)
        ax.axis("off")
        plt.tight_layout()
        plt.show()
        plt.close(fig)

## 8) Kết quả trên tập Test & Val (tại ngưỡng conf* tối ưu)

Khi `eval.auto_eval_test: true`, best checkpoint tự động quét đường cong PR trên tập Test để tìm ngưỡng tin cậy tối ưu `conf*` theo Micro-F1, sau đó dùng chính `conf*` đó để đánh giá đồng thời cả hai tập `Test` và `Val`. Toàn bộ chỉ số được lưu vào `metrics_test.json`, `metrics.json`, `metrics_val.json` và `pr_curve_data.json`. Nếu các file kết quả chưa tồn tại, code cell dưới đây sẽ tự động chạy `tools.run_eval`.

In [ ]:
import json
import subprocess
import pandas as pd

test_metrics_path = RUN_DIR / "metrics_test.json"
legacy_metrics_path = RUN_DIR / "metrics.json"

if not test_metrics_path.exists() and not legacy_metrics_path.exists():
    subprocess.run([
        "python", "-m", "tools.run_eval",
        "--pipeline", PIPELINE,
        "--runtime", RUNTIME,
        "--overrides", str(OVERRIDES_PATH),
        "--data-root", PROCESSED_ROOT,
        "--run-dir", str(RUN_DIR),
    ], check=True)

if test_metrics_path.exists():
    metrics_test = json.loads(test_metrics_path.read_text(encoding="utf-8"))
elif legacy_metrics_path.exists():
    metrics_test = json.loads(legacy_metrics_path.read_text(encoding="utf-8"))
else:
    metrics_test = {}

val_metrics_path = RUN_DIR / "metrics_val.json"
metrics_val = json.loads(val_metrics_path.read_text(encoding="utf-8")) if val_metrics_path.exists() else None

best_conf = metrics_test.get("conf_threshold", metrics_test.get("best_conf_threshold", "N/A"))
iou_thresh = metrics_test.get("iou_threshold", 0.5)
print(f"[EVAL] Ngưỡng tối ưu: conf* = {best_conf}, iou = {iou_thresh}")

priority_keys = [
    "mAP50-95", "mAP50", "mAP75", "AR100",
    "precision", "recall", "f1", "pr_auc",
    "conf_threshold", "iou_threshold", "fps",
    "latency_ms", "tuned_on_split", "split", "pipeline",
]

seen_keys = set(priority_keys)
extra_keys = []
for k in list(metrics_test.keys()) + (list(metrics_val.keys()) if metrics_val else []):
    if k not in seen_keys and k not in extra_keys:
        extra_keys.append(k)

all_keys = [k for k in priority_keys + extra_keys if k in metrics_test or (metrics_val and k in metrics_val)]

data = {"Test": [metrics_test.get(k, "-") for k in all_keys]}
if metrics_val is not None:
    data["Val"] = [metrics_val.get(k, "-") for k in all_keys]

df_eval = pd.DataFrame(data, index=all_keys)
df_eval

## 9) Xem dự đoán trên vài ảnh Test

In [ ]:
import random
import cv2
from ts_alert.config.resolve import get_processed_root
from ts_alert.pipelines import get_pipeline
from ts_alert.runtime import video_io

pipeline = get_pipeline(PIPELINE, cfg)
detector = pipeline.load_detector(pipeline.best_checkpoint(RUN_DIR))
test_imgs = sorted((get_processed_root(cfg) / "images" / "test").glob("*"))
random.seed(cfg["seed"])

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, path in zip(axes.ravel(), random.sample(test_imgs, k=min(6, len(test_imgs)))):
    try:
        frame = video_io.read_image(path)
    except NotImplementedError:
        frame = cv2.imread(str(path))
    for det in detector.predict(frame):
        x1, y1, x2, y2 = map(int, det["bbox"])
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"{det['class_id']} {det['confidence']:.2f}", (x1, max(y1 - 5, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); ax.set_title(path.name, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()


## 10) Đóng gói kết quả để tải về

Trên Kaggle, file nằm trong tab **Output** của notebook sau khi Save Version.

In [ ]:
from ts_alert.runtime.pack import pack_run_dir
print("Đã nén:", pack_run_dir(RUN_DIR))

## 11) Nhận xét (Thành viên 1 — Pipeline YOLO26m)

1. **Kiến trúc mô hình & Tốc độ suy luận**:
   - Kiến trúc YOLO26m mang lại sự cân bằng rất tốt giữa độ chính xác và tốc độ xử lý thời gian thực (> 60 FPS trên GPU T4), đáp ứng yêu cầu suy luận video camera hành trình trực tiếp của hệ thống ADAS.
   - Mô hình tối ưu hóa tốt trên kích thước canvas 640x640 đã letterbox, giảm thiểu việc mất chi tiết ở các biển báo kích thước nhỏ (COCO small objects).

2. **Ảnh hưởng của Online Augmentation**:
   - Khi `ONLINE = False`: Huấn luyện thuần túy trên dữ liệu offline (đã letterbox chuẩn pad 114 và tăng cường lớp thiểu số an toàn), đảm bảo 100% môi trường so sánh công bằng với Faster R-CNN.
   - Khi `ONLINE = True`: Việc bổ sung mosaic và mixup nhẹ giúp mô hình nhận diện tốt hơn các biển báo bị che khuất một phần trong bối cảnh giao thông phức tạp. Chú ý luôn tắt lật ngang (`hflip = False`) vì nhiều biển báo mang tính định hướng bắt buộc (cấm rẽ trái/phải, hướng đi vòng sang phải...).

3. **Hiệu quả xử lý 52 lớp biển báo**:
   - Nhờ bước tiền xử lý offline đã loại bỏ 811 tiny box (< 8px) và augment 274 box cho 40 lớp thiểu số, mô hình không bị hiện tượng sụp đổ gradient ở các lớp hiếm và đạt độ hội tụ ổn định trên toàn bộ 52 lớp.
